In [27]:
import sys

print(sys.executable)

/data/repo-learning/learning/HBand/data-analysis/.venv/bin/python


In [28]:
from pathlib import Path
import json
import pandas as pd
import re

In [29]:
inventory_file = Path("../data/processed/inventory.json")

In [30]:
with inventory_file.open() as f:
    inventory = json.load(f)

inventory

{'raw_data_directory': '/data/repo-learning/learning/HBand/private/raw',
 'backup_count': 1,
 'backups': [{'name': 'phone_backup_2026.08.08',
   'file_count': 14,
   'files': [{'path': 'aMapStyle/style.data',
     'size_bytes': 1370804,
     'extension': '.data'},
    {'path': 'aMapStyle/style_extra.data',
     'size_bytes': 1342,
     'extension': '.data'},
    {'path': 'ble_change.txt', 'size_bytes': 723826, 'extension': '.txt'},
    {'path': 'ble_connect.txt', 'size_bytes': 3086269, 'extension': '.txt'},
    {'path': 'ble_connect_test.txt', 'size_bytes': 92, 'extension': '.txt'},
    {'path': 'ble_write.txt', 'size_bytes': 16773, 'extension': '.txt'},
    {'path': 'bt_connect.txt', 'size_bytes': 1729, 'extension': '.txt'},
    {'path': 'data_upload.txt', 'size_bytes': 152, 'extension': '.txt'},
    {'path': 'dial_transfer.txt', 'size_bytes': 516, 'extension': '.txt'},
    {'path': 'http_recorder.txt', 'size_bytes': 60, 'extension': '.txt'},
    {'path': 'share_20260731041046.jpeg',


In [31]:
inventory.keys()

dict_keys(['raw_data_directory', 'backup_count', 'backups'])

In [32]:
for b in inventory["backups"]:
    print(f"\n=== {b['name']} ===")
    
    for file in b["files"]:
        print(f"{file['path']:35}{file['size_bytes']:>10,} bytes")


=== phone_backup_2026.08.08 ===
aMapStyle/style.data                1,370,804 bytes
aMapStyle/style_extra.data              1,342 bytes
ble_change.txt                        723,826 bytes
ble_connect.txt                     3,086,269 bytes
ble_connect_test.txt                       92 bytes
ble_write.txt                          16,773 bytes
bt_connect.txt                          1,729 bytes
data_upload.txt                           152 bytes
dial_transfer.txt                         516 bytes
http_recorder.txt                          60 bytes
share_20260731041046.jpeg             125,342 bytes
strava.txt                                224 bytes
watch_dog.txt                           4,763 bytes
weather_location.txt                    2,719 bytes


In [33]:
raw_root = Path(inventory["raw_data_directory"])
backup_name = inventory["backups"][0]["name"]
backup_dir = raw_root / backup_name

for name in [
    "ble_change.txt",
    "ble_connect.txt",
    "ble_write.txt",
    "bt_connect.txt",
    "watch_dog.txt",
    "weather_location.txt",
]:
    
    path = backup_dir / name

    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")

    with path.open("r", errors="replace") as f:
        for i, line in enumerate(f):
            if i >= 10:
                break
            print(line.rstrip())




ble_change.txt

2026-08-09 07:05:27||695:-->watch::::::>device_function->A7,00,00,02,01,02,01,00,07,14,01,01,03,00,00,01,00,06,03,01,
2026-08-09 07:05:27||700:-->〖A7〗第01包第01位:『0x00』------>功能:【血压】, 描述: 〖无此功能〗
2026-08-09 07:05:27||702:-->〖A7〗第01包第02位:『0x00』------>功能:【饮酒】, 描述: 〖无此功能〗
2026-08-09 07:05:27||703:-->〖A7〗第01包第03位:『0x02』------>功能:【健康提醒】, 描述: 〖有健康提醒，且本功能模块下的久坐与0xE1指令的久坐互斥，即0xE7指令与0xE1指令互斥〗
2026-08-09 07:05:27||705:-->〖A7〗第01包第04位:『0x01』------>功能:【肤色类型】, 描述: 〖同0的效果，因杰理公版被默认为1，故废弃1的值~~〗
2026-08-09 07:05:27||707:-->〖A7〗第01包第05位:『0x02』------>功能:【微信运动】, 描述: 〖有微信运动，且为杰理的芯片(针对解决了安卓端杰理芯片无法使用微信运动的兼容)〗
2026-08-09 07:05:27||708:-->〖A7〗第01包第06位:『0x01』------>功能:【拍照】, 描述: 〖App打开拍照界面后启动设备拍照页面〗
2026-08-09 07:05:27||709:-->〖A7〗第01包第07位:『0x00』------>功能:【疲劳度】, 描述: 〖无此功能〗
2026-08-09 07:05:27||710:-->〖A7〗第01包第08位:『0x07』------>功能:【血氧】, 描述: 〖全天血氧模拟血氧系列，全天10分钟一个值〗

ble_connect.txt
2026-07-31 03:08:20||292:-->====================app切换到【前台】START====================
--------------------------->|切换到后台的时间：1

In [34]:

records = []

for name in [
    "ble_change.txt",
    "ble_connect.txt",
    "ble_write.txt",
    "bt_connect.txt",
    "watch_dog.txt",
    "weather_location.txt",
]:
    path = backup_dir / name

    with path.open("r", errors="replace") as f:
        for line in f:
            line = line.rstrip()

            if not line:
                continue

            # Extract timestamp when present
            match = re.match(
                r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})",
                line
            )

            timestamp = match.group(1) if match else None

            # Extract the text after the first log separator
            if "||" in line:
                message = line.split("||", 1)[1]
            else:
                message = line

            records.append({
                "timestamp": timestamp,
                "source_file": name,
                "message": message,
            })

events = pd.DataFrame(records)

print("Events:", len(events))
events.head(20)


Events: 30791


,timestamp,source_file,message
0,2026-08-09 07:05:27,ble_change.txt,"695:-->watch::::::>device_function->A7,00,00,0..."
1,2026-08-09 07:05:27,ble_change.txt,"700:-->〖A7〗第01包第01位:『0x00』------>功能:【血压】, 描述: ..."
2,2026-08-09 07:05:27,ble_change.txt,"702:-->〖A7〗第01包第02位:『0x00』------>功能:【饮酒】, 描述: ..."
3,2026-08-09 07:05:27,ble_change.txt,"703:-->〖A7〗第01包第03位:『0x02』------>功能:【健康提醒】, 描述..."
4,2026-08-09 07:05:27,ble_change.txt,"705:-->〖A7〗第01包第04位:『0x01』------>功能:【肤色类型】, 描述..."
5,2026-08-09 07:05:27,ble_change.txt,"707:-->〖A7〗第01包第05位:『0x02』------>功能:【微信运动】, 描述..."
6,2026-08-09 07:05:27,ble_change.txt,"708:-->〖A7〗第01包第06位:『0x01』------>功能:【拍照】, 描述: ..."
7,2026-08-09 07:05:27,ble_change.txt,"709:-->〖A7〗第01包第07位:『0x00』------>功能:【疲劳度】, 描述:..."
8,2026-08-09 07:05:27,ble_change.txt,"710:-->〖A7〗第01包第08位:『0x07』------>功能:【血氧】, 描述: ..."
9,2026-08-09 07:05:27,ble_change.txt,"711:-->〖A7〗第01包第09位:『0x14』------>功能:【信息提醒包数】, ..."


In [35]:
events.groupby("source_file").size().sort_values(ascending=False)

source_file
ble_connect.txt         27546
ble_change.txt           2913
ble_write.txt             148
watch_dog.txt             128
weather_location.txt       35
bt_connect.txt             21
dtype: int64

In [36]:

commands = events[
    events["message"].str.contains(
        r"\b(?:A0|A1|A3|A7|BD|D8|D3|F4)\b",
        regex=True,
        na=False
    )
].copy()

print("Potential protocol events:", len(commands))

commands[[
    "timestamp",
    "source_file",
    "message"
]].head(30)


Potential protocol events: 4492


,timestamp,source_file,message
0,2026-08-09 07:05:27,ble_change.txt,"695:-->watch::::::>device_function->A7,00,00,0..."
1,2026-08-09 07:05:27,ble_change.txt,"700:-->〖A7〗第01包第01位:『0x00』------>功能:【血压】, 描述: ..."
2,2026-08-09 07:05:27,ble_change.txt,"702:-->〖A7〗第01包第02位:『0x00』------>功能:【饮酒】, 描述: ..."
3,2026-08-09 07:05:27,ble_change.txt,"703:-->〖A7〗第01包第03位:『0x02』------>功能:【健康提醒】, 描述..."
4,2026-08-09 07:05:27,ble_change.txt,"705:-->〖A7〗第01包第04位:『0x01』------>功能:【肤色类型】, 描述..."
5,2026-08-09 07:05:27,ble_change.txt,"707:-->〖A7〗第01包第05位:『0x02』------>功能:【微信运动】, 描述..."
6,2026-08-09 07:05:27,ble_change.txt,"708:-->〖A7〗第01包第06位:『0x01』------>功能:【拍照】, 描述: ..."
7,2026-08-09 07:05:27,ble_change.txt,"709:-->〖A7〗第01包第07位:『0x00』------>功能:【疲劳度】, 描述:..."
8,2026-08-09 07:05:27,ble_change.txt,"710:-->〖A7〗第01包第08位:『0x07』------>功能:【血氧】, 描述: ..."
9,2026-08-09 07:05:27,ble_change.txt,"711:-->〖A7〗第01包第09位:『0x14』------>功能:【信息提醒包数】, ..."


In [37]:
events.groupby("source_file").size().sort_values(ascending=False)

source_file
ble_connect.txt         27546
ble_change.txt           2913
ble_write.txt             148
watch_dog.txt             128
weather_location.txt       35
bt_connect.txt             21
dtype: int64

In [38]:

protocol_rows = []

hex_commands = re.compile(
    r'(?<![0-9A-F])'
    r'(A0|A1|A3|A7|B[A-F]|D[0-9A-F]|F[0-9A-F])'
    r'(?:,[0-9A-F]{2})+'
)

for _, row in events.iterrows():
    message = row["message"]

    matches = hex_commands.findall(message)

    for command in matches:
        protocol_rows.append({
            "timestamp": row["timestamp"],
            "source_file": row["source_file"],
            "command": command,
            "message": message,
        })

protocol = pd.DataFrame(protocol_rows)

print("Protocol rows:", len(protocol))
protocol.head(20)

Protocol rows: 2927


,timestamp,source_file,command,message
0,2026-08-09 07:05:27,ble_change.txt,A7,"695:-->watch::::::>device_function->A7,00,00,0..."
1,2026-08-09 07:05:27,ble_change.txt,A7,"722:-->watch::::::>device_function->A7,00,03,0..."
2,2026-08-09 07:05:27,ble_change.txt,A7,"740:-->watch::::::>device_function->A7,02,11,0..."
3,2026-08-09 07:05:27,ble_change.txt,A7,"759:-->watch::::::>device_function->A7,00,00,0..."
4,2026-08-09 07:05:27,ble_change.txt,BD,"817:-->watch::::::>_K_BT_OPERATE->BD,03,01,00,..."
5,2026-08-09 07:05:27,ble_change.txt,A1,825:-->watch::::::>bbc_battery_password_check-...
6,2026-08-09 07:05:28,ble_change.txt,D8,816:-->watch::::::>read_current_sport_step_ope...
7,2026-08-09 07:05:28,ble_change.txt,F4,877:-->watch::::::>change_watch_language_opera...
8,2026-08-09 07:05:28,ble_change.txt,A0,"996:-->watch::::::>read_battery_operate->A0,00..."
9,2026-08-09 07:05:29,ble_change.txt,A3,"115:-->watch::::::>person_info_operate->A3,01,..."


In [39]:
protocol_counts = (
    protocol["command"]
    .value_counts()
    .rename_axis("command")
    .reset_index(name="count")
)

protocol_counts


protocol_counts.head(30)


,command,count
0,DF,2672
1,A1,150
2,D8,76
3,BD,5
4,A3,5
5,A7,4
6,A0,3
7,FF,3
8,F4,2
9,D3,2


In [40]:
df_protocol = protocol[protocol["command"] == "DF"].copy()
other_protocol = protocol[protocol["command"] != "DF"].copy()

print("DF rows:", len(df_protocol))
print("Other rows:", len(other_protocol))




other_protocol["command"].value_counts()






DF rows: 2672
Other rows: 255


command
A1    150
D8     76
BD      5
A3      5
A7      4
A0      3
FF      3
F4      2
D3      2
BA      1
DC      1
F0      1
FD      1
FE      1
Name: count, dtype: int64

In [41]:

df_protocol[[
    "timestamp",
    "source_file",
    "message"
]].head(20).to_string(index=False)



'          timestamp    source_file                                                                                                                                                                                                                                                                                     message\n2026-08-09 07:26:34 ble_change.txt                                                                                                                                                                                                              113:-->====> 当前任务:DF,01,00,00,需要添加一个5s超时的回复检测任务 ==》开启5s超时回复检测:msgWhat = -58621\n2026-08-09 07:26:34 ble_change.txt                                                                                                                                                                                     211:-->watch::::::>head_origal_df_operate->DF,01,00,00,59,00,03,00,00,00,D2,03,52,50,00,00,00,00,00,00,\n2026-08-09 07:26:34 ble_change.txt 212:

In [42]:

df_protocol[[
    "timestamp",
    "message"
]].tail(20).to_string(index=False)


'          timestamp                                                                                                                                                                                                                                                                                     message\n2026-08-09 07:27:34 035:-->watch::::::>head_origal_df_operate->DF,1D,01,01,B1,04,08,07,17,28,B2,0A,00,6F,00,7E,00,60,00,39,35,00,B3,06,00,00,00,00,00,FF,B4,05,00,00,00,00,00,B5,05,00,00,00,00,00,B6,05,FF,FF,FF,FF,FF,B7,33,00,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,\n2026-08-09 07:27:34 092:-->watch::::::>head_origal_df_operate->DF,1D,01,02,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,FF,B8,02,00,00,B9,1E,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,05,40,40,00,00,BA,05,23,0B,09,25,22,BB,1A,00,00,00,00,00,00,00,\n2026-08-09 07:27:34 094:-->watch::::::>head_origal_df_operate->DF,1D,01,03,00,00,00,

In [43]:

df_packets = []

for _, row in events.iterrows():
    message = row["message"]

    # Only actual DF packet lines
    match = re.search(r'->DF,([0-9A-Fa-f,]+)', message)

    if not match:
        continue

    payload = match.group(1)

    # Remove trailing comma if present
    payload = payload.rstrip(",")

    bytes_list = payload.split(",")

    df_packets.append({
        "timestamp": row["timestamp"],
        "source_file": row["source_file"],
        "bytes": bytes_list,
        "length": len(bytes_list),
    })

df = pd.DataFrame(df_packets)

print("Actual DF packets:", len(df))
df.head()


Actual DF packets: 2663


,timestamp,source_file,bytes,length
0,2026-08-09 07:26:34,ble_change.txt,"[01, 00, 00, 59, 00, 03, 00, 00, 00, D2, 03, 5...",19
1,2026-08-09 07:26:34,ble_change.txt,"[01, 00, 01, B1, 04, 08, 09, 00, 00, B2, 0A, 0...",79
2,2026-08-09 07:26:34,ble_change.txt,"[01, 00, 02, FF, FF, FF, FF, FF, FF, FF, FF, F...",79
3,2026-08-09 07:26:34,ble_change.txt,"[01, 00, 03, 00, 00, 00, 00, 00, 00, 00, 00, 0...",79
4,2026-08-09 07:26:34,ble_change.txt,"[02, 00, 00, 59, 00, 03, 00, 00, 00, D2, 77, E...",19


In [44]:
df["b0"] = df["bytes"].apply(lambda x: x[0] if len(x) > 0 else None)
df["b1"] = df["bytes"].apply(lambda x: x[1] if len(x) > 1 else None)
df["b2"] = df["bytes"].apply(lambda x: x[2] if len(x) > 2 else None)

df["fragment"] = df["bytes"].apply(
    lambda x: int(x[3], 16) if len(x) > 3 else None
)

df[[
    "timestamp",
    "b0",
    "b1",
    "b2",
    "fragment",
    "length"
]].head(30)


,timestamp,b0,b1,b2,fragment,length
0,2026-08-09 07:26:34,01,00,00,89,19
1,2026-08-09 07:26:34,01,00,01,177,79
2,2026-08-09 07:26:34,01,00,02,255,79
3,2026-08-09 07:26:34,01,00,03,0,79
4,2026-08-09 07:26:34,02,00,00,89,19
5,2026-08-09 07:26:34,02,00,01,177,79
6,2026-08-09 07:26:34,02,00,02,255,79
7,2026-08-09 07:26:34,02,00,03,0,79
8,2026-08-09 07:26:34,03,00,00,89,19
9,2026-08-09 07:26:34,03,00,01,177,79


In [45]:
fragment_groups = (
    df.groupby(["b1", "b2"])["fragment"]
      .agg(["count", "nunique", "min", "max"])
      .reset_index()
)


fragment_groups.head(30)


,b1,b2,count,nunique,min,max
0,00,00,599,2,32,89
1,00,01,599,1,177,177
2,00,02,599,1,255,255
3,00,03,599,1,0,0
4,01,00,66,1,32,32
5,01,01,66,1,177,177
6,01,02,66,1,255,255
7,01,03,66,1,0,0
8,FF,20,2,1,1,1
9,FF,59,1,1,0,0


In [46]:

df.groupby(["b1", "b2"])["fragment"].apply(
    lambda x: sorted(x.unique())
).head(30)


df.groupby("fragment")["length"].describe()


,count,mean,std,min,25%,50%,75%,max
fragment,,,,,,,,
0,666.0,78.90991,2.324953,19.0,79.0,79.0,79.0,79.0
1,2.0,19.00000,0.000000,19.0,19.0,19.0,19.0,19.0
32,576.0,19.00000,0.000000,19.0,19.0,19.0,19.0,19.0
89,89.0,19.00000,0.000000,19.0,19.0,19.0,19.0,19.0
177,665.0,79.00000,0.000000,79.0,79.0,79.0,79.0,79.0
255,665.0,79.00000,0.000000,79.0,79.0,79.0,79.0,79.0


In [47]:

df.groupby(["b1", "b2"])["fragment"].apply(
    lambda x: sorted(x.unique())
).head(30)


b1  b2
00  00    [32, 89]
    01       [177]
    02       [255]
    03         [0]
01  00        [32]
    01       [177]
    02       [255]
    03         [0]
FF  20         [1]
    59         [0]
Name: fragment, dtype: object